
#Course 1, Week 1: Lakehouse Architecture & Platform

 

### 1. The Data Architecture Evolution

| Era | Architecture | Strengths | Weaknesses |
|-----|-------------|-----------|------------|
| 1980s-2000s | **Data Warehouse** | ACID, schema, BI | Expensive, rigid, no unstructured |
| 2010s | **Data Lake** | Cheap, flexible, any format | No ACID, quality issues, slow BI |
| 2020s+ | **Data Lakehouse** | Best of both | Requires modern platform |

The lakehouse combines warehouse reliability with lake flexibility.


### 2. Lakehouse Key Properties

A data lakehouse provides:
* **ACID transactions** on data lake storage (via Delta Lake)
* **Schema enforcement and evolution** for data quality
* **Direct BI access** to source data (no ETL to warehouse)
* **Unified batch and streaming** in one architecture
* **Open formats** (Parquet + Delta) — no vendor lock-in
* **Governance** via Unity Catalog


### 3. Databricks Lakehouse Platform Components

```
┌──────────────────────────────────────────────────────┐
│                  Unity Catalog (Governance)          │
├──────────────┬──────────────┬────────────────────────┤
│  Databricks  │  Databricks  │    Databricks          │
│  SQL         │  ML/DS       │    Data Engineering    │
├──────────────┴──────────────┴────────────────────────┤
│                  Delta Lake (Storage Layer)          │
├──────────────────────────────────────────────────────┤
│          Apache Spark (Compute Engine)               │
├──────────────────────────────────────────────────────┤
│          Photon (Accelerated Query Engine)           │
└──────────────────────────────────────────────────────┘
```


### 4. Control Plane vs Data Plane

Databricks separates the **control plane** (managed by Databricks) from the
**data plane** (runs in your cloud account):

* **Control Plane:** Workspace UI, job scheduling, notebooks, cluster management
* **Data Plane:** Compute clusters, data storage, actual processing

This separation ensures your data never leaves your cloud account.

### 5. Explore Your Lakehouse Environment

In [0]:
# Verify we're running on Databricks
try:
    print(f"Spark Version: {spark.version}")
    
    # Try the cluster config tag first
    try:
        dbr_version = spark.conf.get('spark.databricks.clusterUsageTags.sparkVersion')
    except Exception:
        # Fall back to the SQL function method if the config is restricted
        dbr_version = spark.sql("SELECT current_version().dbr_version").collect()[0][0]
        
    print(f"Databricks Runtime: {dbr_version}")
    
except NameError:
    print("Not running on Databricks — use Databricks Free Edition to run this notebook")
    print("Sign up at: https://www.databricks.com/")

Spark Version: 4.2.0
Databricks Runtime: 19.5.x-aarch64-photon-scala2.13



### 6. Delta Lake — The Storage Foundation

Delta Lake provides the reliability layer that makes the lakehouse possible:

| Feature | Data Lake (Parquet) | Delta Lake |
|---------|-------------------|------------|
| ACID Transactions | No | Yes |
| Schema Enforcement | No | Yes |
| Time Travel | No | Yes |
| Streaming + Batch | Separate | Unified |


### 7. Photon — Accelerated Queries

* Photon is Databricks' native vectorized query engine written in C++:
* Up to **7x faster** than standard Spark SQL
* Compatible with Spark APIs — no code changes needed
* Automatically used in SQL Warehouses and Photon-enabled clusters


### 8. Key concepts for the Lakehouse Fundamentals Accreditation

1. A data lakehouse combines the reliability of data warehouses with the flexibility of data lakes
2. Delta Lake provides ACID transactions on top of data lake storage
3. Unity Catalog provides unified governance across all data assets
4. The control plane is managed by Databricks; the data plane runs in your cloud
5. Photon accelerates SQL queries without requiring code changes
6. Open formats (Delta/Parquet) prevent vendor lock-in

## Architecture Comparison

EXERCISE: Complete the comparison table by filling in the missing values.

In [0]:

# EXERCISE: Create a DataFrame comparing architectures
# Fill in the missing values (True/False) based on what you learned
architectures = spark.createDataFrame(
    [
        ("Data Warehouse", True, True, False, False, True),
        ("Data Lake", False, False, True, True, False),
        # EXERCISE: Add the Data Lakehouse row with correct values
        ("Data Lakehouse", True, True, True, True, True),
    ],
    ["architecture", "acid_transactions", "schema_enforcement", "unstructured_data", "low_cost_storage", "bi_support"],
)

architectures.show(truncate=False)

+--------------+-----------------+------------------+-----------------+----------------+----------+
|architecture  |acid_transactions|schema_enforcement|unstructured_data|low_cost_storage|bi_support|
+--------------+-----------------+------------------+-----------------+----------------+----------+
|Data Warehouse|true             |true              |false            |false           |true      |
|Data Lake     |false            |false             |true             |true            |false     |
|Data Lakehouse|true             |true              |true             |true            |true      |
+--------------+-----------------+------------------+-----------------+----------------+----------+



## Delta Lake Basics

EXERCISE: Create a simple Delta table and inspect its properties.

In [0]:
# EXERCISE: Create a Delta table with sample data
# Hint: Use spark.createDataFrame() and .write.format("delta")

# Step 1: Create sample data (at least 5 rows with columns: id, name, value)
# YOUR CODE HERE

data = [
    (1, "Alice Smith", 1250.50),
    (2, "Bob Jones", 450.00),
    (3, "Charlie Brown", 10250.75),
    (4, "Diana Prince", 89.99),
    (5, "Evan Wright", 3400.20)
]

df = spark.createDataFrame(data, schema="id INT, name STRING, value DOUBLE")

# Step 2: Write as Delta table named "lab1_lakehouse"
# YOUR CODE HERE

df.write.format("delta").mode("overwrite").saveAsTable("lab1_lakehouse")

# Step 3: Query the table to verify
# YOUR CODE HERE

display(spark.sql("SELECT * FROM lab1_lakehouse"))

id,name,value
1,Alice Smith,1250.5
2,Bob Jones,450.0
3,Charlie Brown,10250.75
4,Diana Prince,89.99
5,Evan Wright,3400.2


## Explore the Transaction Log

EXERCISE: Look at the Delta transaction log to understand how ACID works.

In [0]:
%sql

-- EXERCISE: View the history of your table
-- Hint: DESCRIBE HISTORY <table_name>
-- YOUR CODE HERE

DESCRIBE HISTORY lab1_lakehouse;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-09-01T10:19:14.000Z,78511136629920,delacruzdaniellemarie@yahoo.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1679585587134954),8bb38745-d748-4149-99af-812fd5e794b5,0901-094617-pjq8vn9q-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1376)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13


## Validation

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Spark is running
    try:
        version = spark.version
        checks.append(("Spark environment", True))
    except Exception:
        checks.append(("Spark environment", False))

    # Check 2: Delta table exists
    try:
        df = spark.sql("SELECT * FROM lab1_lakehouse")
        checks.append(("Delta table created", df.count() >= 5))
    except Exception:
        checks.append(("Delta table created", False))

    # Check 3: Architecture comparison has 3 rows
    try:
        checks.append(("Architecture comparison", architectures.count() == 3))
    except Exception:
        checks.append(("Architecture comparison", False))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()


Lab Validation Results:
----------------------------------------
  [PASS] Spark environment
  [PASS] Delta table created
  [PASS] Architecture comparison

All checks passed! Lab complete.


In [0]:
# Clean up
try:
    spark.sql("DROP TABLE IF EXISTS lab1_lakehouse")
    print("Successfully dropped table 'lab1_lakehouse'.")
except Exception as e:
    print(f"Error encountered while trying to drop table 'lab1_lakehouse': {e}")

Successfully dropped table 'lab1_lakehouse'.
